# D1 · M1.1 — LLM Mechanics for Practitioners
### Annotated Parameter-Comparison Notebook

**Meridian Retail Bank — GenAI Operations Assistant, Phase 0 warm-up**

You're a newly onboarded engineer on Meridian Retail Bank's GenAI Operations team. Before anyone
lets you near a production prompt, you need to understand how the model actually behaves under
different settings — temperature, top-p, max tokens, stop sequences — and how model choice trades
off cost, latency and quality.

**This notebook already contains fully working code for every task.** Your job is not to write
the code — it's to **read each block before you run it, understand what it does and why, run it,
and actually look at the real output.** Anil will walk through this live in class, and your
understanding of this exact code will be tested with a live quiz afterward — so don't just click
Run and scroll past the results, read them.

**Core path (required):** Tasks 1–4.
**Stretch path (Diamond tier):** Task 5.

Every code cell that calls the OpenAI API costs a small amount of real money on your API key. The
prompts here are short and the call counts are deliberately kept low — don't loop this notebook
in a `while True`.


## Setup

Your API key comes from the `OPENAI_API_KEY` environment variable — never hard-code it in this
notebook. If you're on the lab VM, this should already be set; if not, ask in the trainer channel
before proceeding, don't paste a key into a cell.


In [4]:
import os
import time
import json
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not set in this environment. "
    "Set it before continuing — never hard-code a key in this notebook."
)

client = OpenAI()
MODEL_PRIMARY = "gpt-4o-mini"   # cheap, fast — used for Tasks 1 and 2
MODEL_COMPARISON = "gpt-4o"     # used alongside MODEL_PRIMARY in Task 3

BANKING_PROMPT = (
    "A Meridian Retail Bank customer writes in: "
    "\"My debit card was declined twice today even though my balance shows sufficient funds. "
    "I need to pay rent this afternoon. What should I do?\" "
    "Draft a short, professional support reply."
)


## Task 1 — Temperature Sweep

**What this code does:** it asks the model the exact same banking question 6 times — twice each
at three different "creativity" settings (`temperature` = 0, 0.7, and 1.0) — and records how long
each call took, how many tokens it used, and what it actually wrote back.

**Why we do this:** reading that "temperature controls randomness" is one sentence. Watching the
actual wording change in front of you, side by side, is what makes it real. Read the code below
line by line — every line has a comment explaining what it's for — then run it.


In [6]:
temperature_runs = []

# We test 3 temperature settings, twice each, so we can see if the pattern actually repeats
# (one weird response could just be luck — two calls at the same setting tells us more).
for temp in [0, 0.7, 1.0]:
    for run_num in range(2):
        start = time.perf_counter()  # start a stopwatch, so we can measure how long the call takes

        # This is the actual call to the model. temperature=temp is the one setting we're testing;
        # everything else stays identical across all 6 calls, on purpose — that's what makes this
        # a fair comparison instead of six unrelated calls.
        response = client.chat.completions.create(
            model=MODEL_PRIMARY,
            messages=[{"role": "user", "content": BANKING_PROMPT}],
            temperature=temp,
        )

        latency = time.perf_counter() - start  # stop the stopwatch

        # Save everything we care about as one record (a dictionary), so all 6 runs can be
        # compared side by side afterward instead of scrolling back through raw output.
        temperature_runs.append({
            "temperature": temp,
            "latency_seconds": latency,
            "prompt_tokens": response.usage.prompt_tokens,
            "completion_tokens": response.usage.completion_tokens,
            "response_text": response.choices[0].message.content,
        })

print(f"Done — {len(temperature_runs)} calls completed.")


Done — 6 calls completed.


**Now actually look at what happened.** Run the cell below to print all 6 responses side by
side. As you read them: do the `temperature=1.0` responses feel more varied from each other than
the `temperature=0` ones? That specific comparison — not the definition of temperature — is what
the live quiz will ask about.


In [17]:
for r in temperature_runs:
    print(f"temperature={r['temperature']}  |  {r['latency_seconds']:.2f}s  |  "
          f"{r['prompt_tokens']} prompt + {r['completion_tokens']} completion tokens")
    print(f"  -> {r['response_text'][:180]}...")
    print()


temperature=0  |  2.49s  |  52 prompt + 158 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us regarding the issues you experienced with your debit card. I understand ho...

temperature=0  |  2.27s  |  52 prompt + 158 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us regarding the issues you experienced with your debit card. I understand ho...

temperature=0.7  |  7.24s  |  52 prompt + 210 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us regarding the issues you experienced with your debit card. I understand ho...

temperature=0.7  |  2.20s  |  52 prompt + 164 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us. I’m sorry to hear about the issues you experienced with your debit

## Task 2 — Sampling Controls: top_p, max_tokens, stop sequences

**What this code does:** four more calls, same banking question, temperature fixed at 0.7 so
we're only changing one thing at a time — `top_p` (twice, at 0.1 and 1.0), `max_tokens` (cut off
at 20), and a `stop` sequence (a period, `"."`, meaning "stop writing the instant you hit this").

**Why we do this:** these three settings are the ones most likely to silently break a real
production reply if nobody's watching for them — especially `stop` and `max_tokens`, which can
truncate an answer with zero warning. Watch for that specifically in the output below.


In [7]:
sampling_runs = []

def run_and_record(variant_name, **kwargs):
    """One call, tagged with a variant name so we can tell the four checks apart afterward."""
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL_PRIMARY,
        messages=[{"role": "user", "content": BANKING_PROMPT}],
        temperature=0.7,
        **kwargs,
    )
    latency = time.perf_counter() - start
    sampling_runs.append({
        "variant": variant_name,
        "latency_seconds": latency,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "response_text": response.choices[0].message.content,
    })

# Check 1: narrow top_p — only the most likely words are ever considered
run_and_record("top_p_0.1", top_p=0.1)

# Check 2: wide top_p — effectively no narrowing at all
run_and_record("top_p_1.0", top_p=1.0)

# Check 3: hard length limit — the response WILL get cut off mid-sentence, that's expected
run_and_record("max_tokens_20", max_tokens=20)

# Check 4: stop sequence — the response ends the instant a "." appears, even mid-thought
run_and_record("stop_period", stop=["."])

print(f"Done — {len(sampling_runs)} calls completed.")


Done — 4 calls completed.


**Look closely at the last two results especially.** Run the cell below, then check: did the
`max_tokens_20` response get cut off somewhere awkward? Did the `stop_period` response end after
just the first sentence, even if there was clearly more to say? That's not a bug in the code —
that's the exact real-world risk these settings create if used carelessly.


In [16]:
for r in sampling_runs:
    print(f"{r['variant']:<16} |  {r['latency_seconds']:.2f}s  |  "
          f"{r['prompt_tokens']} prompt + {r['completion_tokens']} completion tokens")
    print(f"  -> {r['response_text']}")
    print()


top_p_0.1        |  4.09s  |  52 prompt + 207 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us regarding the issues you experienced with your debit card. I understand how important it is to resolve this matter, especially with your rent payment due.

There could be several reasons for the decline, such as a temporary hold on your account or a technical issue. I recommend the following steps:

1. **Check for Holds**: Log into your online banking account to see if there are any holds or alerts that might explain the decline.
2. **Try Again**: If possible, attempt the transaction again, as it may have been a temporary issue.
3. **Contact Us**: If the problem persists, please call our customer support at [Customer Support Phone Number] or visit your nearest branch for immediate assistance.

We are here to help you resolve this issue as quickly as possible. Thank you for your patience.

Best regards,

[Your Name

## Task 3 — Model Trade-off: Cost / Latency / Quality

**What this code does:** the same banking question, sent to two different models —
`MODEL_PRIMARY` (`gpt-4o-mini`, small and cheap) and `MODEL_COMPARISON` (`gpt-4o`, a flagship
model) — so you can compare them on your own real numbers instead of taking anyone's word for it.

**Why we do this:** "which model should I use" is a question every GenAI practitioner gets asked
constantly. The honest answer is always "it depends, measure it" — this is you doing that
measurement, on a real example, for the first time.

Do **not** treat any dollar price as a permanent fact — OpenAI's pricing changes over time. If you
want to reason about actual cost, check the current price on OpenAI's pricing page yourself.


In [10]:
model_comparison_runs = []

for model_name in [MODEL_PRIMARY, MODEL_COMPARISON]:
    start = time.perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": BANKING_PROMPT}],
        temperature=0.7,
    )
    latency = time.perf_counter() - start

    model_comparison_runs.append({
        "model": model_name,
        "latency_seconds": latency,
        "prompt_tokens": response.usage.prompt_tokens,
        "completion_tokens": response.usage.completion_tokens,
        "response_text": response.choices[0].message.content,
    })

print(f"Done — {len(model_comparison_runs)} calls completed.")


Done — 2 calls completed.


In [13]:
for r in model_comparison_runs:
    print(f"{r['model']:<14} |  {r['latency_seconds']:.2f}s  |  "
          f"{r['prompt_tokens']} prompt + {r['completion_tokens']} completion tokens")
    print(f"  -> {r['response_text'][:180]}...")
    print()


gpt-4o-mini    |  3.26s  |  52 prompt + 202 completion tokens
  -> Subject: Assistance with Your Debit Card Issue

Dear [Customer's Name],

Thank you for reaching out to us regarding the issue with your debit card. I apologize for the inconvenienc...

gpt-4o         |  2.93s  |  52 prompt + 228 completion tokens
  -> Subject: Assistance with Declined Debit Card Transactions

Dear [Customer's Name],

Thank you for reaching out to us. I’m sorry to hear about the inconvenience you experienced with...



## Task 4 — Reflect (short, guided — not an essay)

You've now run 12 real calls and read all the output. Answer these three prompts **in your own
words, in 1–2 sentences each**, based on what you actually just saw above — not a general
textbook answer. This is graded on whether it references your real numbers/observations, not on
length.


### Your reflection

1. **Temperature:** Looking at your Task 1 output, what actually changed between the
   `temperature=0` responses and the `temperature=1.0` responses?

   *(your answer)*

2. **Sampling controls:** Which of the four Task 2 checks had the most surprising or risky
   result, and why?

   *(your answer)*

3. **Model choice:** Based on your actual Task 3 latency/token numbers, when would picking
   `MODEL_COMPARISON` over `MODEL_PRIMARY` be worth it for a real Meridian Retail Bank workload —
   and when would it clearly not be?

   *(your answer)*


In [15]:
annotation_text = """
1. Temperature: The temperature parameter controls the randomness of the model's output. A lower temperature (e.g., 0) makes the model more deterministic and focused, while a higher temperature (e.g., 1.0) allows for more creativity and variability in responses.
2. Sampling Controls: The top_p parameter (nucleus sampling) limits the model's token selection to a subset of the most probable tokens, affecting the diversity of responses. A lower top_p value (e.g., 0.1) restricts the model to only the most likely tokens, resulting in more predictable outputs, while a higher value (e.g., 1.0) allows for a wider range of possible tokens, increasing variability.
3. Model Choice: Different models have different capabilities, response styles, and performance characteristics. For example, gpt-4o-mini is designed to be faster and more cost-effective, while gpt-4o may provide more nuanced and detailed responses. The choice of model can significantly impact the quality and style of the output.
"""


## Task 5 — Stretch (Diamond tier): quantify reproducibility

**What this code does:** runs the same banking question 5 times at `temperature=0` and 5 times at
`temperature=1.0` (10 more calls), then uses `difflib.SequenceMatcher` to measure how *similar*
consecutive responses are to each other at each setting — turning "temperature=0 feels more
consistent" from a vibe into an actual number.

Only do this if you've finished Tasks 1–4 with time to spare.


In [14]:
import difflib

reproducibility_variance = {}

def avg_similarity(temp, n=5):
    texts = []
    for _ in range(n):
        response = client.chat.completions.create(
            model=MODEL_PRIMARY,
            messages=[{"role": "user", "content": BANKING_PROMPT}],
            temperature=temp,
        )
        texts.append(response.choices[0].message.content)

    ratios = [
        difflib.SequenceMatcher(None, texts[i], texts[i + 1]).ratio()
        for i in range(len(texts) - 1)
    ]
    return sum(ratios) / len(ratios)

# Leave this cell un-run (reproducibility_variance = {}) if you're skipping the stretch task.
reproducibility_variance = {
    "temp_0_avg_similarity": avg_similarity(0),
    "temp_1_avg_similarity": avg_similarity(1.0),
}
print(reproducibility_variance)


{'temp_0_avg_similarity': 0.898273910582909, 'temp_1_avg_similarity': 0.23404849483013562}


In one sentence below: given these two similarity numbers, would you recommend low or high
temperature for a banking FAQ responder versus a marketing copy generator? This is exactly the
kind of guardrail decision Module M1.3 builds on.

*(your answer)*


## Render the comparison table

This is the module's actual **Expected Output** per the programme syllabus — a short comparison
table of parameter settings versus observed behaviour. Run this before saving.

In [18]:
def print_comparison_table(temperature_runs, sampling_runs, model_comparison_runs):
    rows = []
    for r in temperature_runs:
        rows.append((f"temperature={r['temperature']}", r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))
    for r in sampling_runs:
        rows.append((r['variant'], r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))
    for r in model_comparison_runs:
        rows.append((r['model'], r['latency_seconds'], r['prompt_tokens'], r['completion_tokens']))

    header = f"{'Setting':<28}{'Latency (s)':<14}{'Prompt tok':<12}{'Completion tok':<15}"
    print(header)
    print("-" * len(header))
    for name, latency, ptok, ctok in rows:
        print(f"{name:<28}{latency:<14}{ptok:<12}{ctok:<15}")

print_comparison_table(temperature_runs, sampling_runs, model_comparison_runs)


Setting                     Latency (s)   Prompt tok  Completion tok 
---------------------------------------------------------------------
temperature=0               2.489302599977236252          158            
temperature=0               2.265768800018122452          158            
temperature=0.7             7.23769090001587852          210            
temperature=0.7             2.202352100022835752          164            
temperature=1.0             2.63611970000783952          193            
temperature=1.0             2.959963999979663652          203            
top_p_0.1                   4.08557980001205652          207            
top_p_1.0                   2.50713349998113752          168            
max_tokens_20               0.724330300028668752          20             
stop_period                 1.31393530001514652          23             
gpt-4o-mini                 3.26412469998467752          202            
gpt-4o                      2.933160299988230752    

## Save your results

Run the cell below last. It assembles everything into `m1_1_comparison_results.json` in this
same folder — the autograder (`Day1/autograder/D1_M1.1_LLM_Mechanics_Autograder.py`) reads this
file, not the notebook itself, so make sure this cell actually runs without error before you
consider the lab done.


In [19]:
results = {
    "temperature_runs": temperature_runs,
    "sampling_runs": sampling_runs,
    "model_comparison_runs": model_comparison_runs,
    "annotation": annotation_text,
    "reproducibility_variance": reproducibility_variance,  # {} if you skipped the stretch task
}

with open("m1_1_comparison_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Saved m1_1_comparison_results.json —", len(json.dumps(results)), "bytes")


Saved m1_1_comparison_results.json — 12473 bytes
